In [1]:
from time import time
import pandas as pd
import numpy as np
from collections import OrderedDict
import warnings

import pandas as pd
from CBFV.composition import generate_features

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    AdaBoostClassifier, GradientBoostingClassifier,
    RandomForestClassifier, ExtraTreesClassifier
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC

from sklearn.preprocessing import StandardScaler

In [2]:
def instantiate_model(model_name):
    model = model_name()
    return model

def fit_model(model, X_train, y_train):
    ti = time()
    model = instantiate_model(model)
    model.fit(X_train, y_train)
    fit_time = time() - ti
    return model, fit_time

def append_result_df(df, result_dict):
    df_result_appended = pd.concat([df, pd.DataFrame([result_dict])], ignore_index=True)
    return df_result_appended

def append_model_dict(dic, model_name, model):
    dic[model_name] = model
    return dic

def evaluate_model(model, X, y_act):
    y_pred = model.predict(X)
    acc = accuracy_score(y_act, y_pred)
    f1 = f1_score(y_act, y_pred, average="weighted", zero_division=0)
    precision = precision_score(y_act, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_act, y_pred, average="weighted", zero_division=0)
    return acc, f1, precision, recall

def fit_evaluate_model(model, model_name, X_train, y_train, X_val, y_val):
    model, fit_time = fit_model(model, X_train, y_train)
    acc_train, f1_train, prec_train, rec_train = evaluate_model(model, X_train, y_train)
    acc_val, f1_val, prec_val, rec_val = evaluate_model(model, X_val, y_val)
    result_dict = {
        'model_name': model_name,
        'model_name_pretty': type(model).__name__,
        'model_params': model.get_params(),
        'fit_time': fit_time,
        'acc_train': acc_train,
        'f1_train': f1_train,
        'precision_train': prec_train,
        'recall_train': rec_train,
        'acc_val': acc_val,
        'f1_val': f1_val,
        'precision_val': prec_val,
        'recall_val': rec_val,
    }
    return model, result_dict

# Reading in dataset

In [3]:
df_train = pd.read_csv('control_dataset_splits/ICSD_train_split.csv')
df_test = pd.read_csv('control_dataset_splits/ICSD_test_split.csv')
df_val = pd.read_csv('control_dataset_splits/ICSD_val_split.csv')

In [4]:
df_train["Crystal_System"] = df_train["Crystal_System"].astype("category")
df_test["Crystal_System"] = df_test["Crystal_System"].astype("category")
df_val["Crystal_System"] = df_val["Crystal_System"].astype("category")


df_train["target"] = df_train["Crystal_System"].cat.codes
df_test["target"] = df_test["Crystal_System"].cat.codes
df_val["target"] = df_val["Crystal_System"].cat.codes

label_mapping = dict(enumerate(df_train["Crystal_System"].cat.categories))

In [5]:
rename_dict = {'Chemical': 'formula'}

df_train = df_train.rename(columns=rename_dict)
df_val = df_val.rename(columns=rename_dict)
df_test = df_test.rename(columns=rename_dict)

In [6]:
df_train

,HMS,formula,Temperature,Pressure,Crystal_System,target
0,P 4/m m m,Nd1Ba1Mn1Fe1O5.45,293.0,0.101325,tetragonal,4
1,P 4/m m m,Nd1Ba1Mn1Fe1O5.17,293.0,0.101325,tetragonal,4
2,P 42 m c,Ca1Mn1Ti1.8V0.2O6,293.0,0.101325,tetragonal,4
3,P 42 m c,Ca1Mn1Ti1.4V0.6O6,293.0,0.101325,tetragonal,4
4,P 42 m c,Ca1Mn1Ti1.2V0.8O6,293.0,0.101325,tetragonal,4
...,...,...,...,...,...,...
1612,F d -3 m S,Co1Cd0.4Mn0.5Fe1.1O4,293.0,0.101325,cubic,0
1613,F d -3 m S,Co1Cd0.4Mn0.6Fe1O4,293.0,0.101325,cubic,0
1614,F d -3 m S,Mn0.5Zn0.5Fe1.99Sc0.01O4,293.0,0.101325,cubic,0
1615,F d -3 m S,Mn0.5Zn0.5Fe1.97Sc0.03O4,293.0,0.101325,cubic,0


In [7]:
# CBFV needs 'formula' and 'target' columns only
df_train_cbfv = df_train[['formula', 'target', 'Temperature', 'Pressure']].copy()
df_val_cbfv   = df_val[['formula', 'target', 'Temperature', 'Pressure']].copy()
df_test_cbfv  = df_test[['formula', 'target', 'Temperature', 'Pressure']].copy()

# Generate Magpie features

In [8]:
X_train_unscaled, y_train, formulae_train, skipped_train = generate_features(
    df_train_cbfv, elem_prop='magpie', drop_duplicates=False, extend_features=True, sum_feat=True)
X_val_unscaled, y_val, formulae_val, skipped_val = generate_features(
    df_val_cbfv, elem_prop='magpie', drop_duplicates=False, extend_features=True, sum_feat=True)
X_test_unscaled, y_test, formulae_test, skipped_test = generate_features(
    df_test_cbfv, elem_prop='magpie', drop_duplicates=False, extend_features=True, sum_feat=True)

Processing Input Data: 100%|██████████| 1617/1617 [00:00<00:00, 56749.50it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 1617/1617 [00:00<00:00, 48229.93it/s]


	Creating Pandas Objects...


Processing Input Data: 100%|██████████| 438/438 [00:00<00:00, 70373.69it/s]


...

	Creating Pandas Objects...


Processing Input Data: 100%|██████████| 228/228 [00:00<00:00, 68977.30it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 228/228 [00:00<00:00, 42133.38it/s]

	Creating Pandas Objects...


In [9]:
print(f"Skipped train: {len(skipped_train)}")
print(f"Skipped val:   {len(skipped_val)}")
print(f"Skipped test:  {len(skipped_test)}")

Skipped train: 0
Skipped val:   0
Skipped test:  0


In [10]:
feature_cols = [col for col in X_train_unscaled.columns if col not in ['HMS', 'Crystal_System', 'target']]

print(X_train_unscaled.shape)  # should be (n_samples, n_features)
print(y_train.shape)  # should be (n_samples,)
print(label_mapping)

(1617, 156)
(1617,)
{0: 'cubic', 1: 'hexagonal', 2: 'monoclinic', 3: 'orthorhombic', 4: 'tetragonal', 5: 'triclinic', 6: 'trigonal'}


In [11]:
from sklearn.preprocessing import normalize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_unscaled)  # fit only on train
X_val   = scaler.transform(X_val_unscaled)
X_test  = scaler.transform(X_test_unscaled)

# X_train = normalize(X_train)
# X_val = normalize(X_val)
# X_test = normalize(X_test)

In [12]:
df_classics = pd.DataFrame(columns=[
    'model_name',
    'model_name_pretty',
    'model_params',
    'fit_time',
    'acc_train',
    'f1_train',
    'precision_train',
    'recall_train',
    'acc_val',
    'f1_val',
    'precision_val',
    'recall_val'
])

RANDOM_SEED = 42

classic_model_classes = OrderedDict({
    'dumc': DummyClassifier,
    'lr':   LogisticRegression,
    'abc':  AdaBoostClassifier,
    'gbc':  GradientBoostingClassifier,
    'rfc':  RandomForestClassifier,
    'etc':  ExtraTreesClassifier,
    'svc':  SVC,
    'lsvc': LinearSVC,
    'knc':  KNeighborsClassifier,
})

# Models with seeds and convergence fixes applied
classic_model_names = OrderedDict({
    'dumc': lambda: DummyClassifier(random_state=RANDOM_SEED),
    'lr':   lambda: LogisticRegression(random_state=RANDOM_SEED, max_iter=1000),
    'abc':  lambda: AdaBoostClassifier(random_state=RANDOM_SEED),
    'gbc':  lambda: GradientBoostingClassifier(random_state=RANDOM_SEED),
    'rfc':  lambda: RandomForestClassifier(random_state=RANDOM_SEED),
    'etc':  lambda: ExtraTreesClassifier(random_state=RANDOM_SEED),
    'svc':  lambda: SVC(random_state=RANDOM_SEED),
    'lsvc': lambda: LinearSVC(random_state=RANDOM_SEED, max_iter=5000),
    'knc':  lambda: KNeighborsClassifier(),  # no random_state needed
})

df_classics = pd.DataFrame()
classic_models = OrderedDict()

ti = time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for model_name, model in classic_model_names.items():
        print(f'Now fitting and evaluating model {model_name}: {model}')
        model, result_dict = fit_evaluate_model(
            model, model_name, X_train, y_train, X_val, y_val
        )
        df_classics = append_result_df(df_classics, result_dict)
        classic_models = append_model_dict(classic_models, model_name, model)

dt = time() - ti
print(f'Finished fitting {len(classic_models)} models, total time: {dt:0.2f} s')

Now fitting and evaluating model dumc: <function <lambda> at 0x7db6307c2980>
Now fitting and evaluating model lr: <function <lambda> at 0x7db5b77b8b80>
Now fitting and evaluating model abc: <function <lambda> at 0x7db5b77b8e00>
Now fitting and evaluating model gbc: <function <lambda> at 0x7db5b77b96c0>
Now fitting and evaluating model rfc: <function <lambda> at 0x7db5b77b9620>
Now fitting and evaluating model etc: <function <lambda> at 0x7db5b77b91c0>
Now fitting and evaluating model svc: <function <lambda> at 0x7db5b77b9580>
Now fitting and evaluating model lsvc: <function <lambda> at 0x7db5b77b94e0>
Now fitting and evaluating model knc: <function <lambda> at 0x7db5b77b9760>
Finished fitting 9 models, total time: 34.11 s


In [13]:
# Sort in order of increasing validation
df_classics = df_classics.sort_values('acc_val', ignore_index=True)
df_classics

,model_name,model_name_pretty,model_params,fit_time,acc_train,f1_train,precision_train,recall_train,acc_val,f1_val,precision_val,recall_val
0,dumc,DummyClassifier,"{'constant': None, 'random_state': 42, 'strate...",0.001330,0.313544,0.149686,0.098310,0.313544,0.273973,0.117838,0.075061,0.273973
1,abc,AdaBoostClassifier,"{'algorithm': 'deprecated', 'estimator': None,...",0.690104,0.595547,0.568860,0.550773,0.595547,0.527397,0.489127,0.467789,0.527397
2,lr,LogisticRegression,"{'C': 1.0, 'class_weight': None, 'dual': False...",0.367455,0.753865,0.750530,0.751997,0.753865,0.659817,0.657762,0.659104,0.659817
3,lsvc,LinearSVC,"{'C': 1.0, 'class_weight': None, 'dual': 'auto...",8.696351,0.773036,0.770085,0.772054,0.773036,0.684932,0.681183,0.681878,0.684932
4,svc,SVC,"{'C': 1.0, 'break_ties': False, 'cache_size': ...",0.145359,0.774892,0.771974,0.775933,0.774892,0.687215,0.676662,0.678628,0.687215
5,knc,KNeighborsClassifier,"{'algorithm': 'auto', 'leaf_size': 30, 'metric...",0.001308,0.823748,0.822536,0.823603,0.823748,0.730594,0.727018,0.726635,0.730594
6,gbc,GradientBoostingClassifier,"{'ccp_alpha': 0.0, 'criterion': 'friedman_mse'...",22.324419,0.970315,0.970312,0.970543,0.970315,0.767123,0.765610,0.769131,0.767123
7,rfc,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.765225,0.987631,0.987630,0.987707,0.987631,0.812785,0.811426,0.814791,0.812785
8,etc,ExtraTreesClassifier,"{'bootstrap': False, 'ccp_alpha': 0.0, 'class_...",0.330107,0.987631,0.987617,0.987822,0.987631,0.837900,0.835292,0.836521,0.837900


In [14]:
from tqdm import tqdm
# Combine train and val for final training
X_train_final = np.concatenate((X_train, X_val), axis=0)
y_train_final = np.concatenate((y_train, y_val), axis=0)

seeds = [42, 123, 456, 789, 1024, 2024, 314, 99, 7, 2000]
models_to_evaluate = ['etc', 'rfc', 'lr', 'lsvc']
seed_rows = []

for seed in tqdm(seeds, desc='Seeds'):
    for model_name in tqdm(models_to_evaluate, desc='Models', leave=False):
        
        model_params = df_classics.loc[df_classics['model_name'] == model_name].iloc[0]['model_params'].copy()
        model_params['random_state'] = seed
        
        model = classic_model_classes[model_name](**model_params)
        model.fit(X_train_final, y_train_final)
        
        acc, f1, precision, recall = evaluate_model(model, X_test, y_test)
        
        seed_rows.append({
            'seed':      seed,
            'model':     model_name,
            'test_acc':  acc,
            'test_f1':   f1,
            'test_prec': precision,
            'test_rec':  recall,
        })

seed_df = pd.DataFrame(seed_rows)

summary = seed_df.groupby('model').agg(
    acc_mean=('test_acc', 'mean'),
    acc_std= ('test_acc', 'std'),
    f1_mean= ('test_f1',  'mean'),
    f1_std=  ('test_f1',  'std'),
).round(4)

print(summary)


seed_df['feature_set'] = 'magpie' 
seed_df.to_csv('results/magpie_seed_results.csv', index=False)

Seeds: 100%|██████████| 10/10 [02:07<00:00, 12.77s/it]

       acc_mean  acc_std  f1_mean  f1_std
model                                    
etc      0.8570   0.0075   0.8580  0.0070
lr       0.6491   0.0000   0.6433  0.0000
lsvc     0.6711   0.0000   0.6660  0.0000
rfc      0.8474   0.0054   0.8491  0.0055
